# Week 3, Lab 4 — Crew + tools


In [1]:
WEEK = 'Week 3'
LAB = 'Lab 4 — crew tools'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 4 — crew tools
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

# cfg
llm = LLM(
    model="ollama/qwen2.5:3b",
    base_url="http://localhost:11434/",
)
print("CrewAI LLM ->", cfg)

CrewAI LLM -> {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'qwen2.5:3b'}


In [5]:
from crewai.tools import BaseTool

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Look up a local fact about agentic AI topics."
    def _run(self, topic: str) -> str:
        return lookup_fact(topic)

class CalcTool(BaseTool):
    name: str = "calculator"
    description: str = "Evaluate arithmetic like '12*8+3'."
    def _run(self, expression: str) -> str:
        return calculator(expression)

researcher = Agent(
    role="Researcher",
    goal="Use lookup_fact for topic facts.",
    backstory="Librarian of the local KB.",
    llm=llm,
    tools=[LookupTool()],
)
analyst = Agent(
    role="Analyst",
    goal="Use calculator for any math.",
    backstory="Likes numbers.",
    llm=llm,
    tools=[CalcTool()],
)
t1 = Task(description="What is LangGraph? Use the lookup tool.", expected_output="1-2 sentences grounded in the tool.", agent=researcher)
t2 = Task(description="Compute 45*12+30 with the calculator and include it in a closing sentence.", expected_output="Short recap including the number.", agent=analyst)
print(Crew(agents=[researcher, analyst], tasks=[t1, t2], process=Process.sequential,verbose=True).kickoff())


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: eb65fa1f-d576-40be-9c3a-26932e1dc701                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: What is LangGraph? Use the lookup tool.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/rahul/Documents/V-Align_projects/LLM_Engineering_opensource/agentic_ai_local_models/.venv/lib/python3.14/site
-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Thought: Action: lookup_fact                                                                                   │
│                                                                                                                 │
│  Using Tool: lookup_fact                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "LangGraph"                                                                                         │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analyst                                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The result of 45*12+30 is 570.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 0f80cac2-9e3a-4f8f-8188-17eb50dc4c0e                                                                     │
│  Agent: Analyst                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The result of 45*12+30 is 570.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: eb65fa1f-d576-40be-9c3a-26932e1dc701                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The result of 45*12+30 is 570.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Next:** 3-agent research → draft → review.
